# 01 · Build the invasive / noxious species lists

Builds the species lists for the project from published sources, with names converted to the forms iNaturalist uses.

**Run top to bottom.** Outputs go to `WORKDIR`:

| file | what it is |
|---|---|
| `invasive_sources_long.csv` | one row per species × source × scope (every listing, for comparing a photo's location to where the plant is listed) |
| `invasive_species_summary.csv` | one row per name, all sources collapsed |
| `species_full_bbox.csv` | **hybrid list**, full bbox (31.3–42.0 N, 120.0–103.0 W) |
| `species_desert_sw.csv` | **hybrid list**, species listed in AZ, NM or CA or on the NPS SW watchlist (for iNat place 118341) |

**Hybrid rule** (`classifier_label`):
- `invasive`: non-native and on ≥ 1 ecological list (SWVMA, NPS SW watchlist, Cal-IPC High/Moderate)
- `noxious only`: non-native, on state/USDA noxious lists only
- `native, noxious`: legally listed but native to part of the region, so never labeled invasive

Legal status is kept separately in `noxious_states` and `legal_status`.

Next step: run `02_inat_counts.ipynb` on the two hybrid lists.

> **In this repository.** This is the first step of the data pipeline. It builds the species lists by hand from the published sources below; no iNaturalist calls are made here. Its hybrid lists feed `04_inat_counts.ipynb`, whose ranked output is committed as `data/reference/ranked_desert_sw.csv` and `ranked_full_bbox.csv`. The dataset itself is then built by `src/invasive_clf/data/build_dataset.py`. Run it in Colab (outputs go to `WORKDIR` on Drive) or locally (outputs go to the current folder).

## Setup

In [ ]:
# Where files live. On Colab this mounts your Google Drive so outputs (and the iNat cache)
# survive disconnects; share the folder with your group. Off Colab it just uses the current folder.
import os
WORKDIR = "/content/drive/MyDrive/invasive_plants"   # change to your shared folder

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
except ImportError:
    pass
print("Working in", os.getcwd())

In [ ]:
import csv
from collections import defaultdict, OrderedDict

BBOX = dict(swlat=31.3, swlng=-120.0, nelat=42.0, nelng=-103.0)

## Sources
Each list below is entered **as published** (typos included). The published name is kept in `source_name`, and `NAME_MAP` (further down) converts it to the iNaturalist name. To add a source, add a `SRC` entry and an `add(...)` call.

In [ ]:
# ---------------------------------------------------------------- sources
SRC = {
 "SWVMA": ("SWVMA Arizona's Top Invasive Plant Species", "https://www.swvma.org/invasive-plants-index/"),
 "NPS":   ("NPS Southwest IPMT Targeted Plant Species Watchlist", "https://home.nps.gov/subjects/invasive/sw.htm"),
 "USDA":  ("USDA AMS 2026 State Noxious-Weed Seed Requirements (Federal Seed Act), rev. July 2026",
           "https://www.ams.usda.gov/sites/default/files/media/StateNoxiousWeedsSeedList.pdf"),
 "AZ":    ("Arizona Dept. of Agriculture Official 2023 Noxious Weed List", "https://azdot.gov/sites/default/files/2024-11/Official-2023-Arizona-Noxious-Weed-List.pdf"),
 "CO":    ("Colorado Dept. of Agriculture Noxious Weed Species", "https://ag.colorado.gov/noxious-weed-species"),
 "NM":    ("New Mexico Dept. of Agriculture Noxious Weed List (June 2020)", "https://nmdeptag.nmsu.edu/media/pdf/noxious-weed-memo-and-list-june-2020.pdf"),
 "NV":    ("Nevada NAC 555.010 designated noxious weeds (eff. 2018)", "https://regulations.justia.com/states/nevada/chapter-555/noxious-weeds/classification-of-weeds/section-555-010"),
 "UT":    ("Utah Admin. Code R68-9-3 State Noxious Weeds", "https://www.law.cornell.edu/regulations/utah/Utah-Admin-Code-R68-9-3"),
 "WY":    ("Wyoming Weed & Pest Council Designated List (2018, as cited)", "https://bluearthrenewables.com/wp-content/uploads/2023/04/Two-Rivers-Wind_Appendix-I_Noxious-Weed-and-Pest-Management-Plan-Jan-2023.pdf"),
 "CALIPC":("Cal-IPC Inventory (High/Moderate ratings)", "https://www.cal-ipc.org/plants/inventory/"),
}

# (source key, scope_type, scope, scope_states, category, block of "name|common")
LISTS = []
def add(src, scope_type, scope, states, category, block):
    LISTS.append((src, scope_type, scope, states, category,
                  [l.strip() for l in block.strip().splitlines() if l.strip()]))

### SWVMA: Arizona's top invasive plants (Northern / Central / Southern AZ)

In [ ]:
# ---- SWVMA (Arizona broad regions)
SWVMA_N = """Alhagi maurorum|Camelthorn
Linaria dalmatica|Dalmatian toadflax
Centaurea diffusa|Diffuse knapweed
Dipsacus fullonum|Fuller's teasel
Euphorbia esula|Leafy spurge
Centaurea stoebe [Russian knapweed entry]|Russian knapweed
Elaeagnus angustifolia|Russian olive
Onopordum acanthium|Scotch thistle
Ulmus pumila|Siberian elm
Centaurea stoebe|Spotted knapweed
Bothriochloa ischaemum|Yellow bluestem
Centaurea solstitialis|Yellow starthistle"""
SWVMA_CS = """Searsia lancea|African sumac
Pennisetum ciliaris|Buffelgrass
Sonchus oleraceus|Common sow thistle
Pennisetum setaceum|Fountain grass
Arundo donax|Giant reed
Sisymbrium irio|London rocket
Centaurea melitensis|Malta starthistle
Bromus rebens|Red brome
Kali Tragus|Russian thistle (tumbleweed)
Brassica tournefortii|Sahara mustard
Tamarix ramosissima|Saltcedar
Oncosiphon pilulifer|Stinknet"""
add("SWVMA","AZ subregion","AZ-Northern","AZ","top invasive",SWVMA_N)
add("SWVMA","AZ subregion","AZ-Central","AZ","top invasive",SWVMA_CS)
add("SWVMA","AZ subregion","AZ-Southern","AZ","top invasive",SWVMA_CS)

### NPS Southwest IPMT Targeted Plant Species Watchlist

In [ ]:
# ---- NPS Southwest watchlist
add("NPS","NPS network","NPS Southwest IPMT parks","AZ;CO;NM;OK;TX;UT","watchlist","""
Arundo donax|Giant reed
Bassia scoparia|Mexican fireweed (kochia)
Brassica tournefortii|African mustard
Bromus rubens|Red brome
Bromus tectorum|Cheatgrass
Carduus nutans|Musk thistle
Centaurea melitensis|Malta starthistle
Cirsium arvense|Canada thistle
Conium maculatum|Poison hemlock
Cynoglossum officinale|Houndstongue
Elaeagnus angustifolia|Russian olive
Eragrostis lehmannii|Lehmann lovegrass
Halogeton glomeratus|Halogeton
Lepidium draba|Hoary cress
Lespedeza cuneata|Sericea lespedeza
Linaria vulgaris|Yellow toadflax
Melia azedarach|Chinaberry
Pennisetum ciliare|Buffelgrass
Rhaponticum repens|Russian knapweed
Saccharum ravennae|Ravennagrass
Salsola spp.|Russian thistles
Sorghum halepense|Johnsongrass
Tamarix ramosissima|Tamarisk
Verbascum thapsis|Common mullein
Matthiola parviflora|Small-flowered stock
Melinis repens|Rose Natal grass
Rubus armeniacus|Himalayan blackberry
Oncosiphon piluliferum|Stinknet
Pennisetum setaceum|Fountain grass
Tamarix aphylla|Athel
Volutaria tubuliflora|Desert knapweed""")

### USDA AMS 2026 state noxious-weed seed lists (AZ, CA, CO sections)

In [ ]:
# ---- USDA AMS (Federal Seed Act state lists) AZ / CA / CO prohibited
add("USDA","state","AZ","AZ","prohibited noxious weed seed","""
Convolvulus arvensis|
Bothriochloa ischaemum|
Bromus rubens|
Bromus diandrus|
Orobanche ramosa|
Cenchrus ciliaris|
Alhagi maurorum|
Bromus tectorum|
Ulmus pumila|
Pennisetum setaceum|
Aegilops cylindrica|
Ipomoea tricolor|
Halogeton glomeratus|
Lepidium draba|
Hydrilla verticillata|
Sorghum halepense|
Centaurea diffusa|
Acroptilon repens|
Centaurea stoebe|
Kochia scoparia|
Lythrum salicaria|
Eragrostis lehmanniana|
Cucumis melo var. dudaim|
Ipomoea x leucantha|
Ipomoea triloba|
Ipomoea purpurea|
Ipomoea hederacea|
Brassica nigra|
Brassica tournefortii|
Sinapis arvensis|
Melinis repens|
Elaeagnus angustifolia|
Asphodelus fistulosus|
Tribulus terrestris|
Elymus repens|
Arundo donax|
Euryops subcarnosus|
Peganum harmala|
Tamarix spp.|
Salvinia molesta|
Cenchrus spinifex|
Cenchrus echinatus|
Chondrilla juncea|
Euphorbia esula|
Centaurea melitensis|
Centaurea calcitrapa|
Centaurea solstitialis|
Oncosiphon pilulifer|
Searsia lancea|
Cirsium vulgare|
Cirsium arvense|
Carduus nutans|
Carduus acanthoides|
Onopordum acathium|
Linaria dalmatica|
Ailanthus altissima|
Carrichtera annua|
Eichornia crassipes|
Isatis tinctoria|""")
add("USDA","state","CA","CA","prohibited noxious weed seed","""
Crupina vulgaris|
Acaena novae-zelandiae|
Acaena pallida|
Helianthus ciliaris|
Acaena anserinifolia|
Orobanche ramosa|
Alhagi maurorum|
Tribolium obliterum|
Arctotheca calendula|
Rorippa austriaca|
Parthenium hysterophorus|
Aegilops cylindrica|
Aegilops geniculata|
Galega officinalis|
Halogeton glomeratus|
Lepidium draba|
Lepidium chalepense|
Lepidium appelianum|
Solanum carolinense|
Solanum elaeagnifolium|
Alternanthera sessilis|
Centaurea diffusa|
Centaurea jacea|
Rhaponticum repens|
Centaurea stoebe|
Centaurea virgata|
Cucumis melo var. dudaim|
Lepidium latifolium|
Achnatherum brachychaetum|
Sesbania punicea|
Ononis alopecuroides|
Halimodendron halodendron|
Atriplex amnicola|
Salsola damascena|
Cenchrus incertus|
Cenchrus longispinus|
Cenchrus echinatus|
Sonchus arvensis|
Euphorbia terracina|
Euphorbia virgata|
Rhagadiolus stellatus|
Centaurea iberica|
Cirsium arvense|
Carduus nutans|
Onopordum spp.|
Carduus acanthoides|
Centaurea sulphurea|
Cirsium undulatum|
Carduus crispus|
Carthamus leucocaulos|
Ludwigia hexapetala|
Ludwigia decurrens|
Isatis tinctoria|""")
add("USDA","state","CA","CA","restricted noxious weed seed (CDFA rated)","""
Alternanthera philoxeroides|
Berteroa incana|
Egeria najas|
Zygophyllum fabago|
Convolvulus arvensis|
Araujia sericifera|
Retama monosperma|
Genista monspessulana|
Cytisus scoparius|
Spartium junceum|
Diodia virginiana|
Delairea odorata|
Myosoton aquaticum|
Potentilla recta|
Symphytum asperum|
Spartina anglica|
Spartina densiflora|
Spartina patens|
Spartina alterniflora|
Fatoua villosa|
Zostera japonica|
Nothoscordum gracile|
Brachypodium sylvaticum|
Cabomba caroliniana|
Nymphoides peltata|
Setaria faberi|
Hydrocharis morsus-ranae|
Allium vineale|
Oenothera xenogaura|
Oenothera sinuosus|
Aegilops triuncialis|
Ulex europaeus|
Physalis viscosa|
Physalis virginiana var. sonorae|
Senecio linearifolius|
Peganum harmala|
Hyoscyamus niger|
Hydrilla verticillata|
Sorghum halepense|
Cortaderia jubata|
Acacia paradoxa|
Pennisetum clandestinum|
Hypericum perforatum|
Volutaria tubuliflora|
Fallopia x bohemica|
Fallopia sachalinensis|
Persicaria wallichii|
Fallopia japonica|
Lythrum salicaria|
Tagetes minuta|
Limnophila sessiliflora|
Limnophila indica|
Taeniatherum caput-medusae|
Cucumis myriocarpus|
Mercurialis ambigua|
Prosopis strombulifera|
Viscum album|
Chorispora tenella|
Solanum cardiophyllum|
Solanum lanceolatum|
Solanum dimidiatum|
Solanum marginatum|
Muhlenbergia schreberi|
Cyperus rotundus|
Cyperus esculentus|
Allium paniculatum|
Asphodelus fistulosus|
Lagarosiphon major|
Panicum antidotale|
Sphaerophysa salsula|
Ludwigia peruviana|
Tribulus terrestris|
Elymus repens|
Ambrosia trifida|
Senecio squalidus|
Senecio jacobaea|
Saccharum ravennae|
Arundo donax|
Oryza rufipogon|
Salsola paulsenii|
Salsola tragus|
Salsola collina|
Salvia virgata|
Salvia aethiopis|
Tamarix chinensis|
Tamarix gallica|
Tamarix parviflora|
Tamarix ramosissima|
Salvinia auriculata|
Chondrilla juncea|
Limnobium spongia|
Limnobium laevigatum|
Leptochloa chinensis|
Euphorbia graminea|
Euphorbia oblongata|
Euphorbia serrata|
Euphorbia dendroides|
Hypericum canariense|
Coincya monensis|
Centaurea calcitrapa|
Centaurea solstitialis|
Dittrichia graveolens|
Hygrophila polysperma|
Lepidium coronopus|
Heteropogon contortus|
Cynara cardunculus|
Cirsium vulgare|
Scolymus hispanicus|
Carduus tenuiflorus|
Cirsium japonicum|
Carduus pycnocephalus|
Carthamus baeticus|
Carthamus lanatus|
Cirsium ochrocentrum|
Linaria dalmatica|
Centaurea melitensis|
Ailanthus altissima|
Drymaria cordata|
Nymphaea mexicana|
Ceratopteris thalictroides|
Striga lutea|
Rorippa sylvestris|""")
add("USDA","state","CO","CO","prohibited noxious weed seed","""
Anoda cristata|
Convolvulus arvensis|
Saponaria officinalis|
Alhagi pseudalhagi|
Anthemis arvensis|
Potentilla recta|
Clematis orientalis|
Chrysanthemum leucanthemum|
Aegilops cylindrica|
Halogeton glomeratus|
Solanum carolinense|
Solanum elaeagnifolium|
Cynoglossum officinale|
Sorghum halepense|
Centaurea nigra|
Centaurea diffusa|
Acroptilon repens|
Centaurea maculosa|
Centaurea virgata|
Lythrum salicaria|
Hibiscus trionum|
Taeniatherum caput-medusae|
Panicum miliaceum|
Cyperus esculentus|
Ambrosia tomentosa|
Ambrosia grayi|
Hesperis matronalis|
Peganum harmala|
Salvia aethiopis|
Tamarix parviflora|
Tamarix ramosissima|
Chrondrilla juncea|
Sorghum almum|
Sonchus arvensis|
Euphorbia cyparissias|
Euphorbia esula|
Euphorbia myrsinites|
Hypericum perforatum|
Centaurea solstitialis|
Madia sativa|
Cirsium arvense|
Carduus nutans|
Carduus acanthoides|
Onopordum acanthium|
Onopordum tauricum|
Linaria dalmatica|
Linaria vulgaris|
Cardaria draba|
Cardaria pubescens|
Lepidium latifolium|
Isatis tinctoria|""")

### Arizona Department of Agriculture, 2023 noxious weed list

In [ ]:
# ---- Arizona (ADA Official 2023)
add("AZ","state","AZ","AZ","Class A","""
Peganum harmala|African rue
Cirsium arvense|Canada thistle
Cucumis melo var. dudaim|Dudaim melon
Isatis tinctoria|Dyer's woad
Eichhornia crassipes|Floating water hyacinth
Salvinia molesta|Giant salvinia
Cardaria draba|Globe-podded hoary cress
Hydrilla verticillata|Hydrilla
Euphorbia esula|Leafy spurge
Carduus acanthoides|Plumeless thistle
Lythrum salicaria|Purple loosestrife
Centaurea calcitrapa|Purple starthistle
Elymus repens|Quackgrass
Chondrilla juncea|Rush skeletonweed
Cenchrus echinatus|Southern sandbur
Centaurea stoebe ssp. micranthos|Spotted knapweed
Euryops subcarnosus|Sweet resinbush
Carrichtera annua|Ward's weed
Sinapis arvensis|Wild mustard""")
add("AZ","state","AZ","AZ","Class B","""
Searsia lancea|African sumac
Brassica nigra|Black mustard
Orobanche ramosa|Branched broomrape
Cirsium vulgare|Bull thistle
Alhagi maurorum|Camelthorn
Linaria dalmatica|Dalmatian toadflax
Centaurea diffusa|Diffuse knapweed
Cenchrus spinifex|Field sandbur
Arundo donax|Giant reed
Halogeton glomeratus|Halogeton
Aegilops cylindrica|Jointed goatgrass
Centaurea melitensis|Malta starthistle
Carduus nutans|Musk thistle
Melinis repens|Natal grass
Asphodelus fistulosus|Onionweed
Bromus diandrus|Ripgut brome
Acroptilon repens|Russian knapweed
Elaeagnus angustifolia|Russian olive
Brassica tournefortii|Saharan mustard
Ulmus pumila|Siberian elm
Onopordum acanthium|Scotch thistle
Oncosiphon pilulifer|Stinknet
Bothriochloa ischaemum|Yellow bluestem
Centaurea solstitialis|Yellow starthistle""")
add("AZ","state","AZ","AZ","Class C","""
Pennisetum ciliare|Buffelgrass
Bromus tectorum|Cheatgrass
Convolvulus arvensis|Field bindweed
Pennisetum setaceum|Fountain grass
Ipomoea purpurea|Common morning glory
Ipomoea tricolor|Grannyvine
Ipomoea hederacea|Ivy-leaf morning glory
Sorghum halepense|Johnsongrass
Kochia scoparia|Kochia
Eragrostis lehmanniana|Lehmann lovegrass
Ipomoea triloba|Littlebell
Ipomoea x leucantha|Morning glory
Tribulus terrestris|Puncturevine
Bromus rubens|Red brome
Tamarix spp.|Saltcedar
Ailanthus altissima|Tree of heaven""")

### Colorado noxious weed lists A, B, C and Watch List

In [ ]:
# ---- Colorado
add("CO","state","CO","CO","List A","""
Alhagi pseudalhagi|Camelthorn
Crupina vulgaris|Common crupina
Salvinia molesta|Giant salvinia
Hydrilla verticillata|Hydrilla
Taeniatherum caput-medusae|Medusahead
Myriophyllum aquaticum|Parrotfeather
Centaurea virgata|Squarrose knapweed
Ventenata dubia|Ventenata
Peganum harmala|African rue
Isatis tinctoria|Dyer's woad
Brassica elongata|Elongated mustard
Butomus umbellatus|Flowering rush
Centaurea x moncktonii|Meadow knapweed
Chondrilla juncea|Rush skeletonweed
Senecio jacobaea|Tansy ragwort
Centaurea solstitialis|Yellow starthistle
Euphorbia cyparissias|Cypress spurge
Arundo donax|Giant reed
Epilobium hirsutum|Hairy willow-herb
Fallopia japonica|Japanese knotweed
Fallopia sachalinensis|Giant knotweed
Fallopia x bohemica|Bohemian knotweed
Salvia aethiopis|Mediterranean sage
Euphorbia myrsinites|Myrtle spurge
Hieracium aurantiacum|Orange hawkweed
Lythrum salicaria|Purple loosestrife
Iris pseudacorus|Yellow flag iris""")
add("CO","state","CO","CO","List B","""
Artemisia absinthium|Absinth wormwood
Hyoscyamus niger|Black henbane
Saponaria officinalis|Bouncingbet
Cirsium vulgare|Bull thistle
Cirsium arvense|Canada thistle
Clematis orientalis|Chinese clematis
Tanacetum vulgare|Common tansy
Dipsacus fullonum|Common teasel
Dipsacus laciniatus|Cutleaf teasel
Linaria dalmatica & genistifolia|Dalmatian toadflax
Hesperis matronalis|Dame's rocket
Centaurea diffusa|Diffuse knapweed
Myriophyllum spicatum|Eurasian watermilfoil
Lepidium draba|Hoary cress
Cynoglossum officinale|Houndstongue
Centaurea x psammogena|Hybrid knapweed
Linaria vulgaris x L. dalmatica|Hybrid toadflax
Aegilops cylindrica|Jointed goatgrass
Euphorbia esula|Leafy spurge
Anthemis cotula|Mayweed chamomile
Verbascum blattaria|Moth mullein
Carduus nutans|Musk thistle
Leucanthemum vulgare|Oxeye daisy
Lepidium latifolium|Perennial pepperweed
Carduus acanthoides|Plumeless thistle
Rhaponticum repens|Russian knapweed
Elaeagnus angustifolia|Russian olive
Tamarix chinensis|Saltcedar
Tamarix parviflora|Saltcedar
Tamarix ramosissima|Saltcedar
Tripleurospermum inodorum|Scentless chamomile
Onopordum acanthium|Scotch thistle
Centaurea stoebe ssp. micranthos|Spotted knapweed
Potentilla recta|Sulfur cinquefoil
Carum carvi|Wild caraway
Cyperus esculentus|Yellow nutsedge
Linaria vulgaris|Yellow toadflax""")
add("CO","state","CO","CO","List C","""
Poa bulbosa|Bulbous bluegrass
Cichorium intybus|Chicory
Arctium minus|Common burdock
Verbascum thapsus|Common mullein
Hypericum perforatum|Common St. Johnswort
Bromus tectorum|Cheatgrass
Convolvulus arvensis|Field bindweed
Halogeton glomeratus|Halogeton
Sorghum halepense|Johnsongrass
Sonchus arvensis|Perennial sowthistle
Conium maculatum|Poison hemlock
Tribulus terrestris|Puncturevine
Elymus repens|Quackgrass
Erodium cicutarium|Redstem filaree
Ulmus pumila|Siberian elm
Ailanthus altissima|Tree of heaven
Abutilon theophrasti|Velvetleaf
Panicum miliaceum|Wild-proso millet""")
add("CO","state","CO","CO","Watch List","""
Gypsophila paniculata|Baby's breath
Bothriochloa bladhii|Caucasian bluestem
Anchusa officinalis|Common bugloss
Phragmites australis|Common reed
Alliaria petiolata|Garlic mustard
Lysimachia vulgaris|Garden loosestrife
Rubus armeniacus|Himalayan blackberry
Berteroa incana|Hoary alyssum
Hieracium caespitosum|Meadow hawkweed
Asphodelus fistulosus|Onionweed
Lathyrus latifolius|Perennial sweet pea
Cytisus scoparius|Scotch broom
Sphaerophysa salsula|Swainsonpea
Zygophyllum fabago|Syrian beancaper
Arrhenatherum elatius|Tall oatgrass
Bryonia alba|White bryony
Bothriochloa ischaemum|Yellow bluestem
Reseda lutea|Yellow mignonette""")

### New Mexico noxious weed list (June 2020)

In [ ]:
# ---- New Mexico
add("NM","state","NM","NM","Class A","""
Hyoscyamus niger|Black henbane
Alhagi psuedalhagi|Camelthorn
Cirsium arvense|Canada thistle
Linaria dalmatica|Dalmatian toadflax
Centaurea diffusa|Diffuse knapweed
Isatis tinctoria|Dyer's woad
Salvinia molesta|Giant salvinia
Cardaria spp.|Hoary cress
Euphorbia esula|Leafy spurge
Leucanthemum vulgare|Oxeye daisy
Lythrum salicaria|Purple loosestrife
Centaurea calcitrapa|Purple starthistle
Saccharum ravennae|Ravenna grass
Matricaria perforata|Scentless chamomile
Onopordum acanthium|Scotch thistle
Centaurea biebersteinii|Spotted knapweed
Centaurea solstitialis|Yellow starthistle
Linaria vulgaris|Yellow toadflax""")
add("NM","state","NM","NM","Class B","""
Peganum harmala|African rue
Cirsium vulgare|Bull thistle
Cichorium intybus|Chicory
Halogeton glomeratus|Halogeton
Centaurea melitensis|Malta starthistle
Lepidium latifolium|Perennial pepperweed
Conium maculatum|Poison hemlock
Elytrigia repens|Quackgrass
Xanthium spinosum|Spiny cocklebur
Dipsacus fullonum|Teasel""")
add("NM","state","NM","NM","Class C","""
Bromus tectorum|Cheatgrass
Potamogeton crispus|Curlyleaf pondweed
Myriophyllum spicatum|Eurasian watermilfoil
Arundo donax|Giant cane
Hydrilla verticllata|Hydrilla
Aegilops cylindrica|Jointed goatgrass
Carduus nutans|Musk thistle
Myriophyllum aquaticum|Parrotfeather
Acroptilon repans|Russian knapweed
Elaeagnus angustifolia|Russian olive
Tamarix spp.|Saltcedar
Ulmus pumila|Siberian elm
Ailanthus altissima|Tree of heaven""")
add("NM","state","NM","NM","Watch List","""
Pennisetum ciliaris|Buffelgrass
Pennisetum setaceum|Crimson fountaingrass
Centaurea pratensis|Meadow knapweed
Euphorbia myrsinites|Myrtle spurge
Cortaderia sellonana|Pampas grass
Bothriochloa ischaemum|Yellow bluestem""")

### Nevada NAC 555.010

In [ ]:
# ---- Nevada
add("NV","state","NV","NV","Category A","""
Peganum harmala|African rue
Rorippa austriaca|Austrian fieldcress
Sphaerophysa salsula|Austrian peaweed
Aegilops triuncialis|Barbed goatgrass
Pennisetum ciliare|Buffelgrass
Alhagi pseudalhagi|Camelthorn
Crupina vulgaris|Common crupina
Potamogton crispus|Curlyleaf pondweed
Volutaria tubuliflora|Desert knapweed
Isatis tinctoria|Dyer's woad
Myriophyllum spicatum|Eurasian watermilfoil
Butomus umbellatus|Flowering rush
Salvinia molesta|Giant salvinia
Galega officinalis|Goat's rue
Pennisetum setaceum|Green fountain grass
Cynoglossum officinale|Houndstongue
Hydrilla verticillata|Hydrilla
Centaurea iberica|Iberian starthistle
Aegilops cylindrica|Jointed goatgrass
Hypericum perforatum|Klamath weed
Centaurea melitensis|Malta starthistle
Salvia aethiopis|Mediterranean sage
Lythrum salicaria|Purple loosestrife
Lythrum virgatum|Purple loosestrife
Centaurea calcitrapa|Purple starthistle
Chondrilla juncea|Rush skeletonweed
Centaurea virgata|Squarrose knapweed
Potentilla recta|Sulfur cinquefoil
Zygophyllum fabago|Syrian bean caper
Ventenata dubia|Ventenata
Centaurea solstitialis|Yellow starthistle
Linaria vulgaris|Yellow toadflax""")
add("NV","state","NV","NV","Category B","""
Hyoscyamus niger|Black henbane
Solanum carolinense|Carolina horsenettle
Linaria dalmatica|Dalmatian toadflax
Centaurea diffusa|Diffuse knapweed
Arundo donax|Giant reed
Euphorbia esula|Leafy spurge
Anthemis cotula|Mayweed chamomile
Taeniatherum caput-medusae|Medusahead
Sonchus arvensis|Perennial sowthistle
Brassica tournefortii|Sahara mustard
Centaurea maculosa|Spotted knapweed
Solanum elaeagnifolium|White horsenettle""")
add("NV","state","NV","NV","Category C","""
Cirsium arvense|Canada thistle
Cardaria draba|Hoary cress
Sorghum halepense|Johnsongrass
Carduus nutans|Musk thistle
Lepidium latifolium|Perennial pepperweed
Conium maculatum|Poison hemlock
Tribulus terrestris|Puncturevine
Acroptilon repens|Russian knapweed
Tamarix spp.|Saltcedar
Onopordum acanthium|Scotch thistle
Cicuta maculata|Water hemlock""")

### Utah R68-9-3

In [ ]:
# ---- Utah
add("UT","state","UT","UT","Class 1A (EDRR watch)","""
Crupina vulgaris|Common crupina
Salvia aethiopis|Mediterranean sage
Carduus acanthoides|Plumeless thistle
Anchusa arvensis|Small bugloss
Milium vernale|Spring millet
Zygophyllum fabago|Syrian bean caper""")
add("UT","state","UT","UT","Class 1B (EDRR)","""
Brassica tournefortii|African mustard
Peganum harmala|African rue
Echium vulgare|Blueweed (viper's bugloss)
Hypericum perforatum|Common St. Johnswort
Scorzonera laciniata|Cutleaf vipergrass
Brassicas elongate|Elongated mustard
Myriophyllum spicatum|Eurasian watermilfoil
Alliaria petiolate|Garlic mustard
Arundo donax|Giant reed
Galega officinalis|Goat's rue
Polygonum cuspidatum|Japanese knotweed
Centaurea melitensis|Malta starthistle
Leucanthemum vulgare|Oxeye daisy
Myriophyllum aquaticum|Parrotfeather
Centaurea calcitrapa|Purple starthistle
Ventenata dubia|Ventenata
Alhagi maurorum|Camelthorn""")
add("UT","state","UT","UT","Class 2 (control)","""
Hyoscyamus niger|Black henbane
Linaria dalmatica|Dalmatian toadflax
Centaurea diffusa|Diffuse knapweed
Isatis tinctoria|Dyer's woad
Euphorbia esula|Leafy spurge
Taeniatherum caput-medusae|Medusahead
Lythrum salicaria|Purple loosestrife
Chondrilla juncea|Rush skeletonweed
Centaurea stoebe|Spotted knapweed
Centaurea virgata|Squarrose knapweed
Centaurea solstitialis|Yellow starthistle
Linaria vulgaris|Yellow toadflax""")
add("UT","state","UT","UT","Class 3 (containment)","""
Cynodon dactylon|Bermudagrass
Cirsium arvense|Canada thistle
Convolvulus spp.|Field bindweed
Lepidium draba|Hoary cress (globe-podded)
Lepidium chalepense|Hoary cress (heart-podded)
Lepidium appelianum|Hoary cress (lens-podded)
Cynoglossum officianale|Houndstongue
Aegilops cylindrica|Jointed goatgrass
Carduus nutans|Musk thistle
Lepidium latifolium|Perennial pepperweed
Sorghum halepense|Johnsongrass
Sorghum almum|Columbus grass
Phragmites australis ssp.|Phragmites (common reed)
Conium maculatum|Poison hemlock
Tribulus terrestris|Puncturevine
Elymus repens|Quackgrass
Acroptilon repens|Russian knapweed
Onopordum acanthium|Scotch thistle
Tamarix ramosissima|Tamarisk (saltcedar)""")
add("UT","state","UT","UT","Class 4 (prohibited sale)","""
Imperata cylindrica|Cogongrass
Hesperis matronalis|Dame's rocket
Euphorbia myrsinites|Myrtle spurge
Elaeagnus angustifolia|Russian olive
Cytisus scoparius|Scotch broom""")

### Wyoming designated list (2018, as cited). Only the 41–42° N strip is in the box

In [ ]:
# ---- Wyoming (only lat 41-42 strip is in the box)
add("WY","state","WY","WY","designated noxious","""
Convolvulus arvensis|Field bindweed
Cirsium arvense|Canada thistle
Euphorbia esula|Leafy spurge
Sonchus arvensis|Perennial sowthistle
Agropyron repens|Quackgrass
Cardaria draba|Hoary cress (whitetop)
Cardaria pubescens|Hoary cress (whitetop)
Lepidium latifolium|Perennial pepperweed
Chrysanthemum leucanthemum|Oxeye daisy
Franseria discolor|Skeletonleaf bursage
Centaurea repens|Russian knapweed
Linaria vulgaris|Yellow toadflax
Linaria dalmatica|Dalmatian toadflax
Onopordum acanthium|Scotch thistle
Carduus nutans|Musk thistle
Arctium minus|Common burdock
Carduus acanthoides|Plumeless thistle
Isatis tinctoria|Dyer's woad
Cynoglossum officinale|Houndstongue
Centaurea maculosa|Spotted knapweed
Centaurea diffusa|Diffuse knapweed
Lythrum salicaria|Purple loosestrife
Tamarix spp.|Saltcedar
Hypericum perforatum|Common St. Johnswort
Tanacetum vulgare|Common tansy
Elaeagnus angustifolia|Russian olive
Hyoscyamus niger|Black henbane
Verbascum thapsus|Common mullein
Centaurea solstitialis|Yellow starthistle
Ventenata dubia|Ventenata
Taeniatherum caput-medusae|Medusahead""")

### Cal-IPC Inventory, High and Moderate

In [ ]:
# ---- Cal-IPC
add("CALIPC","state","CA","CA","High","""
Aegilops triuncialis|Barb goatgrass
Alternanthera philoxeroides|Alligatorweed
Ammophila arenaria|European beachgrass
Arundo donax|Giant reed
Brassica tournefortii|Sahara mustard
Bromus madritensis ssp. rubens|Red brome
Bromus tectorum|Cheatgrass
Carpobrotus edulis|Highway iceplant
Carthamus lanatus|Woolly distaff thistle
Centaurea solstitialis|Yellow starthistle
Centaurea stoebe ssp. australis|Spotted knapweed
Cortaderia jubata|Jubatagrass
Cortaderia selloana|Pampasgrass
Cytisus scoparius|Scotch broom
Delairea odorata|Cape ivy
Egeria densa|Brazilian waterweed
Ehrharta calycina|Veldtgrass
Eichhornia crassipes|Water hyacinth
Elymus caput-medusae|Medusahead
Euphorbia virgata|Leafy spurge (virgata)
Genista monspessulana|French broom
Hedera canariensis|Algerian ivy
Hedera helix|English ivy
Hydrilla verticillata|Hydrilla
Lepidium latifolium|Perennial pepperweed
Limnobium spongia|American spongeplant
Ludwigia hexapetala|Uruguay water-primrose
Ludwigia peploides|Floating water-primrose
Lythrum salicaria|Purple loosestrife
Myriophyllum aquaticum|Parrotfeather
Myriophyllum spicatum|Eurasian watermilfoil
Oncosiphon pilulifer|Stinknet
Onopordum acanthium|Scotch thistle
Rubus armeniacus|Himalayan blackberry
Salvinia molesta|Giant salvinia
Sesbania punicea|Red sesbania
Spartina alterniflora x S. foliosa|Hybrid cordgrass
Spartina densiflora|Dense-flowered cordgrass
Spartium junceum|Spanish broom
Tamarix chinensis|Five-stamen tamarisk
Tamarix gallica|French tamarisk
Tamarix parviflora|Smallflower tamarisk
Tamarix ramosissima|Saltcedar
Ulex europaeus|Gorse""")
add("CALIPC","state","CA","CA","Moderate","""
Acacia dealbata|Silver wattle
Ageratina adenophora|Croftonweed
Ailanthus altissima|Tree of heaven
Alhagi maurorum|Camelthorn
Alliaria petiolata|Garlic mustard
Arctotheca calendula|Fertile capeweed
Arctotheca prostrata|Capeweed
Asparagus asparagoides|Bridal creeper
Asphodelus fistulosus|Onionweed
Atriplex semibaccata|Australian saltbush
Avena barbata|Slender oat
Avena fatua|Wild oat
Brachypodium distachyon|Purple false brome
Brachypodium sylvaticum|Slender false-brome
Brassica nigra|Black mustard
Bromus diandrus|Ripgut brome
Carduus nutans|Musk thistle
Carduus pycnocephalus|Italian thistle
Carpobrotus chilensis|Sea fig
Carrichtera annua|Ward's weed
Centaurea calcitrapa|Purple starthistle
Centaurea diffusa|Diffuse knapweed
Centaurea jacea ssp. pratensis|Meadow knapweed
Centaurea melitensis|Malta starthistle (tocalote)
Centaurea virgata ssp. squarrosa|Squarrose knapweed
Chondrilla juncea|Rush skeletonweed
Chrysanthemoides monilifera|Boneseed
Cirsium arvense|Canada thistle
Cirsium vulgare|Bull thistle
Clematis vitalba|Old man's beard
Colocasia esculenta|Taro
Conium maculatum|Poison hemlock
Cotoneaster franchetii|Orange cotoneaster
Cotoneaster lacteus|Milkflower cotoneaster
Cotoneaster pannosus|Silverleaf cotoneaster
Cynara cardunculus|Artichoke thistle
Cynodon dactylon|Bermudagrass
Cynoglossum officinale|Houndstongue
Cynosurus echinatus|Hedgehog dogtail
Cytisus striatus|Portuguese broom
Dipsacus fullonum|Fuller's teasel
Dipsacus sativus|Indian teasel
Dittrichia graveolens|Stinkwort
Ehrharta erecta|Panic veldtgrass
Elaeagnus angustifolia|Russian olive
Emex spinosa|Devil's thorn
Fallopia japonica|Japanese knotweed
Fallopia sachalinensis|Giant knotweed
Festuca arundinacea|Tall fescue
Festuca myuros|Rattail fescue
Festuca perennis|Italian ryegrass
Ficus carica|Edible fig
Foeniculum vulgare|Fennel
Gazania linearis|Gazania
Genista monosperma|Bridal broom
Glyceria declinata|Waxy mannagrass
Halogeton glomeratus|Halogeton
Hirschfeldia incana|Shortpod mustard
Holcus lanatus|Velvetgrass
Hordeum marinum|Mediterranean barley
Hordeum murinum|Wall barley
Hypericum canariense|Canary Island St. Johnswort
Hypochaeris radicata|Rough cat's-ear
Isatis tinctoria|Dyer's woad
Lepidium chalepense|Lens-podded whitetop
Lepidium draba|Hoary cress
Leucanthemum vulgare|Oxeye daisy
Limonium duriusculum|European sea lavender
Linaria dalmatica ssp. dalmatica|Dalmatian toadflax
Linaria vulgaris|Yellow toadflax
Lythrum hyssopifolia|Hyssop loosestrife
Mentha pulegium|Pennyroyal
Mesembryanthemum crystallinum|Crystalline iceplant
Myoporum laetum|Myoporum
Nicotiana glauca|Tree tobacco
Oxalis pes-caprae|Bermuda buttercup
Pennisetum setaceum|Fountain grass
Phalaris aquatica|Harding grass
Potamogeton crispus|Curlyleaf pondweed
Rhaponticum repens|Russian knapweed
Rumex acetosella|Sheep sorrel
Saccharum ravennae|Ravennagrass
Salsola soda|Oppositeleaf Russian thistle
Schinus terebinthifolius|Brazilian peppertree
Senecio glomeratus|Cutleaf burnweed
Spartina anglica|Common cordgrass
Stipa capensis|Mediterranean steppegrass
Tanacetum vulgare|Common tansy
Torilis arvensis|Tall sock-destroyer
Triadica sebifera|Chinese tallow
Vinca major|Big periwinkle
Washingtonia robusta|Mexican fan palm
Zostera japonica|Dwarf eelgrass""")

## Name mapping and flags
- `NAME_MAP`: published name → iNaturalist name (anything not listed passes through unchanged)
- `CHECK`: names whose iNaturalist form is uncertain (`name_status = check`)
- `NATIVE_FLAG`: listed somewhere but native to part of the region
- `AQUATIC`: aquatic/wetland species, removed from the hybrid lists
- `COMMON_FILL`: common names for USDA rows, which give none

In [ ]:
# ---------------------------------------------------------------- names
# published name -> name used by iNaturalist. Anything not listed passes through unchanged.
NAME_MAP = {
 "Pennisetum ciliaris":"Cenchrus ciliaris","Pennisetum ciliare":"Cenchrus ciliaris",
 "Pennisetum setaceum":"Cenchrus setaceus","Pennisetum clandestinum":"Cenchrus clandestinus",
 "Bromus rebens":"Bromus rubens","Bromus madritensis ssp. rubens":"Bromus rubens",
 "Kali Tragus":"Salsola tragus","Salsola spp.":"Salsola",
 "Centaurea stoebe [Russian knapweed entry]":"Rhaponticum repens",
 "Acroptilon repens":"Rhaponticum repens","Acroptilon repans":"Rhaponticum repens","Centaurea repens":"Rhaponticum repens",
 "Centaurea maculosa":"Centaurea stoebe","Centaurea biebersteinii":"Centaurea stoebe",
 "Centaurea stoebe ssp. micranthos":"Centaurea stoebe","Centaurea stoebe ssp. australis":"Centaurea stoebe",
 "Alhagi pseudalhagi":"Alhagi maurorum","Alhagi psuedalhagi":"Alhagi maurorum",
 "Kochia scoparia":"Bassia scoparia",
 "Cardaria draba":"Lepidium draba","Cardaria spp.":"Lepidium draba",
 "Cardaria pubescens":"Lepidium appelianum","Cardaria chalepensis":"Lepidium chalepense",
 "Elytrigia repens":"Elymus repens","Agropyron repens":"Elymus repens",
 "Chrysanthemum leucanthemum":"Leucanthemum vulgare",
 "Matricaria perforata":"Tripleurospermum inodorum",
 "Franseria discolor":"Ambrosia tomentosa",
 "Linaria dalmatica & genistifolia":"Linaria dalmatica","Linaria dalmatica ssp. dalmatica":"Linaria dalmatica",
 "Eichornia crassipes":"Pontederia crassipes","Eichhornia crassipes":"Pontederia crassipes",
 "Cucumis melo var. dudaim":"Cucumis melo",
 "Hydrilla verticllata":"Hydrilla verticillata","Potamogton crispus":"Potamogeton crispus",
 "Onopordum acathium":"Onopordum acanthium","Chrondrilla juncea":"Chondrilla juncea",
 "Verbascum thapsis":"Verbascum thapsus","Cynoglossum officianale":"Cynoglossum officinale",
 "Brassicas elongate":"Brassica elongata","Alliaria petiolate":"Alliaria petiolata",
 "Polygonum cuspidatum":"Reynoutria japonica","Fallopia japonica":"Reynoutria japonica",
 "Fallopia sachalinensis":"Reynoutria sachalinensis","Fallopia x bohemica":"Reynoutria × bohemica",
 "Cortaderia sellonana":"Cortaderia selloana",
 "Elymus caput-medusae":"Taeniatherum caput-medusae",
 "Centaurea pratensis":"Centaurea × moncktonii","Centaurea jacea ssp. pratensis":"Centaurea × moncktonii",
 "Centaurea x moncktonii":"Centaurea × moncktonii","Centaurea x psammogena":"Centaurea × psammogena",
 "Centaurea virgata ssp. squarrosa":"Centaurea virgata",
 "Oncosiphon piluliferum":"Oncosiphon pilulifer",
 "Eragrostis lehmannii":"Eragrostis lehmanniana",
 "Senecio jacobaea":"Jacobaea vulgaris",
 "Lythrum virgatum":"Lythrum salicaria",
 "Tamarix spp.":"Tamarix","Convolvulus spp.":"Convolvulus arvensis",
 "Phragmites australis ssp.":"Phragmites australis",
 "Linaria vulgaris x L. dalmatica":"Linaria vulgaris × dalmatica",
 "Ipomoea x leucantha":"Ipomoea × leucantha","Sorghum almum":"Sorghum × almum",
 "Cenchrus incertus":"Cenchrus spinifex",
 "Genista monosperma":"Retama monosperma",
 "Chrysanthemoides monilifera":"Chrysanthemoides monilifera",
 "Onopordum spp.":"Onopordum",
 "Spartina alterniflora":"Sporobolus alterniflorus","Spartina anglica":"Sporobolus anglicus",
 "Spartina densiflora":"Sporobolus densiflorus","Spartina patens":"Sporobolus pumilus",
 "Spartina alterniflora x S. foliosa":"Sporobolus alterniflorus × foliosus",
 "Achnatherum brachychaetum":"Amelichloa brachychaeta",
 "Festuca perennis":"Lolium multiflorum",
 "Salsola damascena":"Caroxylon vermiculatum",
}
# names whose iNaturalist form I'm not certain of -> verify first
CHECK = {"Salsola tragus","Salsola","Centaurea × moncktonii","Centaurea × psammogena","Linaria vulgaris × dalmatica",
         "Ipomoea × leucantha","Sorghum × almum","Festuca arundinacea","Festuca myuros","Lolium multiflorum",
         "Retama monosperma","Stipa capensis","Caroxylon vermiculatum","Amelichloa brachychaeta",
         "Sporobolus alterniflorus","Sporobolus anglicus","Sporobolus densiflorus","Sporobolus pumilus",
         "Sporobolus alterniflorus × foliosus","Oenothera xenogaura","Oenothera sinuosus",
         "Physalis virginiana var. sonorae","Leptochloa chinensis","Cucumis melo","Bromus rubens",
         "Pontederia crassipes","Reynoutria × bohemica","Chrysanthemoides monilifera"}

# Native to part of the bbox (or native N. American) despite being listed somewhere -> flag

In [ ]:
NATIVE_FLAG = {
 "Solanum elaeagnifolium":"native to the SW US", "Helianthus ciliaris":"native to the SW US",
 "Cirsium undulatum":"native to western US","Cirsium ochrocentrum":"native to SW/Great Plains",
 "Ambrosia tomentosa":"native to western US","Ambrosia grayi":"native to Great Plains",
 "Cicuta maculata":"native to North America","Phragmites australis":"has a native subspecies (americanus)",
 "Cyperus esculentus":"native varieties in North America","Nymphaea mexicana":"native to the SW US",
 "Solanum carolinense":"native to eastern US","Muhlenbergia schreberi":"native to eastern US",
 "Cenchrus spinifex":"native to the Americas","Cenchrus longispinus":"native to North America",
 "Cenchrus echinatus":"native to the Americas","Ipomoea purpurea":"native to Mexico",
 "Ipomoea hederacea":"native to tropical America","Solanum dimidiatum":"native to TX/OK",
 "Solanum lanceolatum":"native to Mexico","Oenothera xenogaura":"native to TX",
 "Oenothera sinuosus":"native to S Great Plains","Physalis virginiana var. sonorae":"native to North America",
 "Physalis viscosa":"native to the Americas","Heteropogon contortus":"native in the SW US",
 "Sporobolus alterniflorus":"native to Atlantic coast","Sporobolus pumilus":"native to Atlantic coast",
 "Diodia virginiana":"native to eastern US","Washingtonia robusta":"native to NW Mexico (Baja/Sonora)",
 "Cynodon dactylon":"introduced, but cultivated turf ubiquitous", "Xanthium spinosum":"status varies by source",
}
AQUATIC = {"Hydrilla verticillata","Salvinia molesta","Salvinia auriculata","Pontederia crassipes","Myriophyllum spicatum",
 "Myriophyllum aquaticum","Egeria densa","Egeria najas","Potamogeton crispus","Butomus umbellatus","Limnobium spongia",
 "Limnobium laevigatum","Ludwigia hexapetala","Ludwigia peploides","Ludwigia peruviana","Ludwigia decurrens",
 "Cabomba caroliniana","Hydrocharis morsus-ranae","Nymphoides peltata","Lagarosiphon major","Limnophila sessiliflora",
 "Limnophila indica","Hygrophila polysperma","Zostera japonica","Ceratopteris thalictroides","Nymphaea mexicana",
 "Alternanthera philoxeroides","Sporobolus alterniflorus","Sporobolus anglicus","Sporobolus densiflorus",
 "Sporobolus pumilus","Sporobolus alterniflorus × foliosus","Iris pseudacorus","Oryza rufipogon"}

COMMON_FILL = {  # common names for USDA rows (that source gives none); standard USDA/Jepson common names
 "Acacia paradoxa":"Kangaroo thorn","Acaena anserinifolia":"Bidgee-widgee","Acaena novae-zelandiae":"Biddy-biddy",
 "Acaena pallida":"Pale biddy-biddy","Aegilops geniculata":"Ovate goatgrass","Allium paniculatum":"Panicled onion",
 "Allium vineale":"Wild garlic","Alternanthera sessilis":"Sessile joyweed","Ambrosia grayi":"Woollyleaf bur ragweed",
 "Ambrosia trifida":"Giant ragweed","Amelichloa brachychaeta":"Puna needlegrass","Anoda cristata":"Spurred anoda",
 "Anthemis arvensis":"Corn chamomile","Araujia sericifera":"Bladderflower","Atriplex amnicola":"Swamp saltbush",
 "Cabomba caroliniana":"Fanwort","Carduus crispus":"Curly plumeless thistle","Carduus tenuiflorus":"Slenderflower thistle",
 "Caroxylon vermiculatum":"Mediterranean saltwort","Carthamus baeticus":"Smooth distaff thistle",
 "Carthamus leucocaulos":"Whitestem distaff thistle","Cenchrus clandestinus":"Kikuyugrass",
 "Cenchrus longispinus":"Longspine sandbur","Centaurea jacea":"Brown knapweed","Centaurea nigra":"Black knapweed",
 "Centaurea sulphurea":"Sicilian starthistle","Ceratopteris thalictroides":"Water sprite",
 "Chorispora tenella":"Blue mustard","Cirsium japonicum":"Japanese thistle","Cirsium ochrocentrum":"Yellowspine thistle",
 "Cirsium undulatum":"Wavyleaf thistle","Coincya monensis":"Star mustard","Cucumis myriocarpus":"Paddy melon",
 "Cyperus rotundus":"Purple nutsedge","Diodia virginiana":"Virginia buttonweed","Drymaria cordata":"Whitesnow",
 "Egeria najas":"Najas waterweed","Euphorbia dendroides":"Tree spurge","Euphorbia graminea":"Grassleaf spurge",
 "Euphorbia oblongata":"Oblong spurge","Euphorbia serrata":"Serrate spurge","Euphorbia terracina":"Geraldton carnation weed",
 "Fatoua villosa":"Hairy crabweed","Halimodendron halodendron":"Russian salt tree","Helianthus ciliaris":"Texas blueweed",
 "Heteropogon contortus":"Tanglehead","Hibiscus trionum":"Flower-of-an-hour","Hydrocharis morsus-ranae":"European frog-bit",
 "Hygrophila polysperma":"Indian swampweed","Lagarosiphon major":"African elodea","Lepidium coronopus":"Swinecress",
 "Leptochloa chinensis":"Chinese sprangletop","Limnobium laevigatum":"South American spongeplant",
 "Limnophila indica":"Indian marshweed","Limnophila sessiliflora":"Asian marshweed","Ludwigia decurrens":"Wingleaf primrose-willow",
 "Ludwigia peruviana":"Peruvian primrose-willow","Madia sativa":"Coast tarweed","Mercurialis ambigua":"Annual mercury",
 "Muhlenbergia schreberi":"Nimblewill","Myosoton aquaticum":"Giant chickweed","Nothoscordum gracile":"Slender false garlic",
 "Nymphaea mexicana":"Yellow waterlily","Nymphoides peltata":"Yellow floatingheart","Oenothera sinuosus":"Wavyleaf gaura",
 "Oenothera xenogaura":"Drummond's gaura","Ononis alopecuroides":"Foxtail restharrow","Onopordum":"Cotton thistles",
 "Onopordum tauricum":"Taurian thistle","Oryza rufipogon":"Brownbeard rice","Panicum antidotale":"Blue panicgrass",
 "Parthenium hysterophorus":"Santa Maria feverfew","Persicaria wallichii":"Himalayan knotweed",
 "Physalis virginiana var. sonorae":"Longleaf groundcherry","Physalis viscosa":"Starhair groundcherry",
 "Prosopis strombulifera":"Argentine screwbean","Rhagadiolus stellatus":"Endive daisy","Rorippa sylvestris":"Creeping yellowcress",
 "Salsola collina":"Slender Russian thistle","Salsola paulsenii":"Barbwire Russian thistle","Salvia virgata":"Meadow sage",
 "Salvinia auriculata":"Eared watermoss","Scolymus hispanicus":"Golden thistle","Senecio linearifolius":"Fireweed groundsel",
 "Senecio squalidus":"Oxford ragwort","Setaria faberi":"Giant foxtail","Solanum cardiophyllum":"Heartleaf nightshade",
 "Solanum dimidiatum":"Western horsenettle","Solanum lanceolatum":"Lanceleaf nightshade","Solanum marginatum":"Purple African nightshade",
 "Sporobolus alterniflorus":"Smooth cordgrass","Sporobolus pumilus":"Saltmeadow cordgrass","Striga lutea":"Witchweed",
 "Symphytum asperum":"Rough comfrey","Tagetes minuta":"Wild marigold","Tribolium obliterum":"Canary Island grass",
 "Viscum album":"European mistletoe",
}

## Build the long and summary files

In [ ]:
def main():
    long_rows = []
    common_seen = {}
    for src, stype, scope, states, cat, lines in LISTS:
        for ln in lines:
            pub, _, common = ln.partition("|")
            pub = pub.strip(); common = common.strip()
            inat = NAME_MAP.get(pub, pub)
            if common and inat not in common_seen:
                common_seen[inat] = common
            rank = "genus" if " " not in inat else ("hybrid" if "×" in inat else "species")
            long_rows.append(OrderedDict(
                inat_name=inat, common_name=common, source_name=pub, rank=rank,
                source=SRC[src][0], source_url=SRC[src][1], list_category=cat,
                scope_type=stype, scope=scope, scope_states=states))
    for r in long_rows:
        if not r["common_name"]:
            r["common_name"] = common_seen.get(r["inat_name"], COMMON_FILL.get(r["inat_name"], ""))
        r["name_status"] = "check" if (r["inat_name"] in CHECK or r["rank"] != "species") else "ok"
        r["native_flag"] = NATIVE_FLAG.get(r["inat_name"], "")
        r["aquatic"] = "yes" if r["inat_name"] in AQUATIC else ""

    long_rows.sort(key=lambda r: (r["inat_name"], r["scope"], r["source"]))
    with open("invasive_sources_long.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(long_rows[0].keys())); w.writeheader(); w.writerows(long_rows)

    agg = defaultdict(lambda: dict(srcs=set(), scopes=set(), states=set(), names=set(), cats=set()))
    for r in long_rows:
        a = agg[r["inat_name"]]
        a["common"] = r["common_name"]; a["rank"] = r["rank"]; a["status"] = r["name_status"]
        a["native"] = r["native_flag"]; a["aquatic"] = r["aquatic"]
        a["srcs"].add(r["source"].split(" ")[0] if not r["source"].startswith("USDA") else "USDA")
        a["scopes"].add(r["scope"]); a["names"].add(r["source_name"])
        a.setdefault("commons", []).append(r["common_name"])
        if r["scope_type"] != "NPS network":          # count only state/subregion listings
            a["states"].update(r["scope_states"].split(";"))
        else:
            a["nps"] = "yes"
        a["cats"].add(f'{r["scope"]}: {r["list_category"]}')
    summ = []
    for name, a in agg.items():
        commons = [c for c in a["commons"] if c]
        common = max(sorted(set(commons)), key=commons.count) if commons else ""
        summ.append(OrderedDict(
            inat_name=name, common_name=common, rank=a["rank"], name_status=a["status"],
            n_sources=len(a["srcs"]), sources=";".join(sorted(a["srcs"])),
            states_listed=";".join(sorted(a["states"])), n_states=len(a["states"]),
            nps_sw_watchlist=a.get("nps", ""),
            scopes=";".join(sorted(a["scopes"])), categories=" | ".join(sorted(a["cats"])),
            source_names=";".join(sorted(a["names"])), native_flag=a["native"], aquatic=a["aquatic"]))
    summ.sort(key=lambda r: (-r["n_states"], -r["n_sources"], r["inat_name"]))
    with open("invasive_species_summary.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(summ[0].keys())); w.writeheader(); w.writerows(summ)
    print(len(long_rows), "long rows;", len(summ), "taxa")
    print("missing common names:", [s["inat_name"] for s in summ if not s["common_name"]])

main()

## Build the hybrid lists
Cleanup carried over from earlier decisions: aquatic species removed; Bermudagrass and Mexican fan palm removed; *Salsola* spp. merged into *Salsola tragus*. Genus-level legal listings (*Tamarix* spp., *Onopordum* spp.) are copied to the species of that genus on the list.

If `invasive_ecological_ranked.csv` from an earlier counts run is in `WORKDIR`, its taxon ids are reused so notebook 02 can skip those lookups. Without it, the `taxon_id` column is left blank and notebook 02 looks the names up.

In [ ]:
import pandas as pd
from collections import OrderedDict

ECO = ("SWVMA", "NPS", "Cal-IPC")
DROP = {"Cynodon dactylon", "Washingtonia robusta"}
GENUS_ROWS = {"Tamarix", "Onopordum", "Salsola"}
DESERT_SW_STATES = {"AZ", "NM", "CA"}
# lists often call every tamarisk "saltcedar"; use the standard names so classes stay distinct
COMMON_OVERRIDE = {"Tamarix ramosissima": "Saltcedar", "Tamarix chinensis": "Five-stamen tamarisk",
                   "Tamarix parviflora": "Smallflower tamarisk", "Tamarix gallica": "French tamarisk",
                   "Tamarix aphylla": "Athel"}

def short_source(s):
    for k in ECO:
        if s.startswith(k): return k
    if s.startswith("USDA"): return "USDA-AMS"
    for st, k in [("Arizona", "AZ-ADA"), ("Colorado", "CO-CDA"), ("New Mexico", "NM-NMDA"), ("Nevada", "NV-NAC"),
                  ("Utah", "UT-UDAF"), ("Wyoming", "WY-WWPC")]:
        if s.startswith(st): return k
    return s

L = pd.read_csv("invasive_sources_long.csv", keep_default_na=False)
L["src"] = L.source.map(short_source)
L["list_type"] = L.src.map(lambda s: "ecological" if s in ECO else "regulatory")

# ---- fixes to names
L.loc[L.source_name == "Cucumis melo var. dudaim", "inat_name"] = "Cucumis melo var. dudaim"  # not all melons
L.loc[L.inat_name == "Salsola", "inat_name"] = "Salsola tragus"
L = L[(L.aquatic != "yes") & ~L.inat_name.isin(DROP)]

# ---- push genus-level listings down to the species of that genus already on the list
species_names = set(L.inat_name)
extra = []
for g in ("Tamarix", "Onopordum"):
    members = sorted(n for n in species_names if n.startswith(g + " "))
    for _, r in L[L.inat_name == g].iterrows():
        for m in members:
            rr = r.copy(); rr["inat_name"] = m; rr["common_name"] = ""; rr["list_category"] += f" ({g} spp.)"; extra.append(rr)
L = pd.concat([L[~L.inat_name.isin({"Tamarix", "Onopordum"})], pd.DataFrame(extra)], ignore_index=True)
L = L.drop_duplicates(["inat_name", "src", "scope", "list_category"])

# ---- taxon ids already resolved in the last iNat run
ids = {}
if os.path.exists("invasive_ecological_ranked.csv"):
    R = pd.read_csv("invasive_ecological_ranked.csv", keep_default_na=False)
    ids = {r.inat_name: (r.taxon_id, r.inat_accepted_name) for r in R.itertuples() if str(r.taxon_id).strip()}
print(len(ids), "taxon ids reused from an earlier run")

def j(vals, sep=";"):
    return sep.join(sorted({v for v in vals if v}))

In [ ]:
rows = []
for name, g in L.groupby("inat_name"):
    eco, reg = g[g.list_type == "ecological"], g[g.list_type == "regulatory"]
    native = j(g.native_flag)
    label = "native, noxious" if native else ("invasive" if len(eco) else "noxious only")
    eco_states = set(); [eco_states.update(s.split(";")) for s in eco.scope_states]
    nox_states = set(); [nox_states.update(s.split(";")) for s in reg.scope_states]
    legal = {}
    for r in reg.itertuples():
        legal.setdefault(r.scope, set()).add(f"{r.list_category} [{r.src}]")
    commons = [c for c in g.common_name if c]
    tid, acc = ids.get(name, ("", ""))
    rank = "hybrid" if "×" in name else ("variety" if " var. " in name else "species")
    status = "check" if (rank != "species" or "check" in set(g.name_status)) else "ok"
    rows.append(OrderedDict(
        inat_name=name, common_name=COMMON_OVERRIDE.get(name) or (max(sorted(set(commons)), key=commons.count) if commons else ""),
        rank=rank, name_status=status,
        classifier_label=label, native_flag=native,
        ecological_sources=j(eco.src), ecological_scopes=j(eco.scope), ecological_states=j(eco_states),
        ecological_detail="; ".join(sorted({f"{r.scope} ({r.src} {r.list_category})" for r in eco.itertuples()})),
        noxious_states=j(nox_states), n_noxious_states=len(nox_states),
        legal_status=" | ".join(f"{st}: " + "; ".join(sorted(v)) for st, v in sorted(legal.items())),
        all_states=j(eco_states | nox_states),
        source_names=j(g.source_name), source_urls=j(g.source_url),
        taxon_id=tid, inat_accepted_name=acc))
S = pd.DataFrame(rows).sort_values(["classifier_label", "inat_name"])
S.to_csv("species_full_bbox.csv", index=False)

in_sw = S.all_states.map(lambda s: bool(DESERT_SW_STATES & set(s.split(";"))))
D = S[in_sw].copy()
D["desert_sw_scope"] = D.apply(lambda r: j(DESERT_SW_STATES & set(r.all_states.split(";"))), axis=1)
D.to_csv("species_desert_sw.csv", index=False)

for nm, df in (("full bbox", S), ("desert SW", D)):
    print(f"{nm}: {len(df)} species", df.classifier_label.value_counts().to_dict(),
          "| need iNat lookup:", (df.taxon_id == "").sum(), "| name check:", (df.name_status == "check").sum())
print("not in desert SW list:", sorted(set(S.inat_name) - set(D.inat_name)))

## Look at the results

In [ ]:
pd.set_option("display.max_colwidth", 80)
cols = ["inat_name", "common_name", "classifier_label", "ecological_sources", "noxious_states", "legal_status"]
display(D[cols].sort_values(["classifier_label", "inat_name"]).head(25))
D.classifier_label.value_counts()

Optional: download the hybrid lists. They're already saved in `WORKDIR` on Drive.

In [ ]:
try:
    from google.colab import files
    for f in ("species_full_bbox.csv", "species_desert_sw.csv"):
        files.download(f)
except ImportError:
    print("Not on Colab: files are in", os.getcwd())